# Get citing & cited opinions metadata for State Courts

In the previous experiments, we've primarily focused on SCOTUS. This is to expand the dataset to all courts in the State jurisdiction, including the State highest courts, appeals courts, and district courts. 

This notebook documents the steps I undertook to:
1. Identify the State courts to include
2. Use Django Shell to sample target cases in the target courts from CL Replica
3. Use Django Shell to get all cases that cited the sampled target cases - these are the citing cases
4. Use Django Shell to get all authorities for the citing cases - these are the cited cases, including ones in the specified courts (target cases) and the ones not in the specified courts
5. Use Django Shell to get related metadata for the citing and cited cases

# Import Libaries

In [1]:
import json

import numpy as np
import pandas as pd

# Load Court Hierarchy data

In [2]:
df = pd.read_csv("../experiments_624/court_hierarchy_flp.csv")
df.head()

,id,full_name,jurisdiction,jurisdiction_name,jurisdiction_type,jurisdiction_state,appeals_to_full_name,appeals_to_id,note
0,scotus,Supreme Court of the United States,F,Federal Appellate,Federal,Federal,NaN,NaN,NaN
1,cafc,Court of Appeals for the Federal Circuit,F,Federal Appellate,Federal,Federal,Supreme Court of the United States,scotus,NaN
2,ca1,Court of Appeals for the First Circuit,F,Federal Appellate,Federal,Federal,Supreme Court of the United States,scotus,NaN
3,ca2,Court of Appeals for the Second Circuit,F,Federal Appellate,Federal,Federal,Supreme Court of the United States,scotus,NaN
4,ca3,Court of Appeals for the Third Circuit,F,Federal Appellate,Federal,Federal,Supreme Court of the United States,scotus,NaN


In [3]:
df["jurisdiction_name"].value_counts()

jurisdiction_name
Federal Bankruptcy          141
Federal District             94
State Appellate              57
State Supreme                52
Federal Appellate            14
Federal Bankruptcy Panel      8
State Trial                   6
Territory Supreme             5
Territory Trial               4
Territory Appellate           2
Name: count, dtype: int64

In [4]:
state_supreme = df[df["jurisdiction_name"] == "State Supreme"]
state_supreme["full_name"]

257                 Supreme Court of Alabama
260                     Alaska Supreme Court
262                    Arizona Supreme Court
264                Supreme Court of Arkansas
266                 California Supreme Court
269                Supreme Court of Colorado
271             Supreme Court of Connecticut
274                Supreme Court of Delaware
277    District of Columbia Court of Appeals
278                 Supreme Court of Florida
280                 Supreme Court of Georgia
282                     Hawaii Supreme Court
284                      Idaho Supreme Court
286                   Illinois Supreme Court
288                    Indiana Supreme Court
290                    Supreme Court of Iowa
292                  Supreme Court of Kansas
294                   Kentucky Supreme Court
296               Supreme Court of Louisiana
298          Supreme Judicial Court of Maine
300             Court of Appeals of Maryland
302     Massachusetts Supreme Judicial Court
306       

In [5]:
state_appeals = df[df["jurisdiction_name"] == "State Appellate"]
state_appeals["full_name"]

258                 Court of Criminal Appeals of Alabama
259                    Court of Civil Appeals of Alabama
261                           Court of Appeals of Alaska
263                          Court of Appeals of Arizona
265                         Court of Appeals of Arkansas
267                           California Court of Appeal
268    Appellate Division of the Superior Court of Ca...
270                            Colorado Court of Appeals
272                          Connecticut Appellate Court
273                           Connecticut Superior Court
275                        Court of Chancery of Delaware
276                           Superior Court of Delaware
279                  District Court of Appeal of Florida
281                          Court of Appeals of Georgia
283                 Hawaii Intermediate Court of Appeals
285                               Idaho Court of Appeals
287                          Appellate Court of Illinois
289                            

In [6]:
state_trial = df[df["jurisdiction_name"] == "State Trial"]
state_trial["full_name"]

299            Superior Court of Maine
320    Superior Court of New Hampshire
328             New York Supreme Court
329             New York County Courts
330            New York District Court
331             New York Justice Court
Name: full_name, dtype: object

In [7]:
state_appeals["jurisdiction_state"].value_counts()

jurisdiction_state
Massachusetts     3
Alabama           2
New York          2
California        2
North Carolina    2
Connecticut       2
Delaware          2
Texas             2
Tennessee         2
Oklahoma          2
Pennsylvania      2
Oregon            1
New Mexico        1
North Dakota      1
Ohio              1
South Carolina    1
Rhode Island      1
Nevada            1
Utah              1
Vermont           1
Virginia          1
Washington        1
West Virginia     1
New Jersey        1
Mississippi       1
Nebraska          1
Missouri          1
Arizona           1
Arkansas          1
Colorado          1
Florida           1
Georgia           1
Hawaii            1
Idaho             1
Illinois          1
Indiana           1
Iowa              1
Kansas            1
Kentucky          1
Louisiana         1
Maryland          1
Michigan          1
Minnesota         1
Alaska            1
Wisconsin         1
Name: count, dtype: int64

In [8]:
court_ids = state_supreme["id"].to_list() + state_appeals["id"].to_list() + state_trial["id"].to_list()
len(court_ids)

115

In [9]:
court_ids

['ala',
 'alaska',
 'ariz',
 'ark',
 'cal',
 'colo',
 'conn',
 'del',
 'dc',
 'fla',
 'ga',
 'haw',
 'idaho',
 'ill',
 'ind',
 'iowa',
 'kan',
 'ky',
 'la',
 'me',
 'md',
 'mass',
 'mich',
 'minn',
 'miss',
 'mo',
 'mont',
 'neb',
 'nev',
 'nh',
 'nj',
 'nm',
 'ny',
 'nc',
 'nd',
 'ohio',
 'okla',
 'or',
 'pa',
 'ri',
 'sc',
 'sd',
 'tenn',
 'tex',
 'texcrimapp',
 'utah',
 'vt',
 'va',
 'wash',
 'wva',
 'wis',
 'wyo',
 'alacrimapp',
 'alacivapp',
 'alaskactapp',
 'arizctapp',
 'arkctapp',
 'calctapp',
 'calappdeptsuper',
 'coloctapp',
 'connappct',
 'connsuperct',
 'delch',
 'delsuperct',
 'fladistctapp',
 'gactapp',
 'hawapp',
 'idahoctapp',
 'illappct',
 'indctapp',
 'iowactapp',
 'kanctapp',
 'kyctapp',
 'lactapp',
 'mdctspecapp',
 'massappct',
 'masssuperct',
 'massdistct',
 'michctapp',
 'minnctapp',
 'missctapp',
 'moctapp',
 'nebctapp',
 'nevapp',
 'njsuperctappdiv',
 'nmctapp',
 'nyappdiv',
 'nyappterm',
 'ncctapp',
 'ncsuperct',
 'ndctapp',
 'ohioctapp',
 'oklacivapp',
 'oklac

# Use Django Shell to get the list of target case cluster ids and citing case cluster ids

In [10]:
with open("data/citing_ids.txt", "r") as f:
    citing_ids = [line.strip() for line in f]
len(citing_ids)

1210

In [11]:
with open("data/target_ids.txt", "r") as f:
    target_ids = [int(line.strip()) for line in f]
len(target_ids)

197

## Ensure the citing cases are not already part of the SCOTUS set in review

In [12]:
scotus = pd.read_json("../experiments_624/data/scotus_citing_cited_sampled.json")
len(scotus)

21925

In [13]:
scotus_citing = list(set(scotus["citing_cluster_id"].to_list()))
len(scotus_citing)

482

In [14]:
assert len(set(citing_ids) - set(scotus_citing)) == len(citing_ids)

# Use Django Shell to get the citing cases metadata

In [15]:
with open('data/citing_opinions.json', 'r') as f:
    results = json.load(f)

In [16]:
records = []
for case_id, case_data in results.items():
    record = {'citing_cluster_id': int(case_id)}
    record.update({k: v for k, v in case_data.items()})
    
    opinion_filenames = [op['opinion_filename'] for op in case_data.get('opinion_data', [])]
    record['opinion_filenames'] = opinion_filenames
    
    records.append(record)

citing_df = pd.DataFrame(records)
citing_df.head()

,citing_cluster_id,citing_url,citing_court_id,citing_court_name,opinion_data,cited_cluster_ids,opinion_filenames
0,217242,https://www.courtlistener.com/opinion/217242/d...,ca1,Court of Appeals for the First Circuit,"[{'opinion_id': 217242, 'opinion_api': None, '...","[2526, 46231, 153651, 179285, 199916, 200574, ...",[217242_010combined.txt]
1,278557,https://www.courtlistener.com/opinion/278557/l...,ca8,Court of Appeals for the Eighth Circuit,"[{'opinion_id': 278557, 'opinion_api': None, '...","[1480682, 1482910, 1490695, 1503605, 1791903, ...",[278557_010combined.txt]
2,530570,https://www.courtlistener.com/opinion/530570/j...,ca5,Court of Appeals for the Fifth Circuit,"[{'opinion_id': 530570, 'opinion_api': None, '...","[86058, 94318, 97670, 103255, 110092, 111563, ...",[530570_010combined.txt]
3,886834,https://www.courtlistener.com/opinion/886834/s...,mont,Montana Supreme Court,"[{'opinion_id': 886834, 'opinion_api': None, '...","[885084, 885303, 885670, 111170, 886075]",[886834_010combined.txt]
4,888975,https://www.courtlistener.com/opinion/888975/s...,mont,Montana Supreme Court,"[{'opinion_id': 888975, 'opinion_api': None, '...","[101031, 106545, 876343, 876921, 879716, 88032...",[888975_010combined.txt]


## Get all cited case ids to a list for extracting the metadata

In [17]:
#with open('data/cited_ids.txt', 'w') as f:
#    for each in list(set(citing_df["cited_cluster_ids"].explode().dropna().tolist())):
#        f.write(f"{each}\n")

# Use Django Shell to extract the metadata for the cited cases

In [18]:
with open('data/cited_opinions.json', 'r') as f:
    results = json.load(f)

In [19]:
cited_metadata = pd.DataFrame.from_dict(results, orient='index')
cited_metadata = cited_metadata.reset_index().rename(columns={'index': 'cited_cluster_id'})
cited_metadata["cited_cluster_id"] = cited_metadata["cited_cluster_id"].astype(int)
cited_metadata.head()

,cited_cluster_id,cited_url,cited_court_id,cited_court_name,cited_case_name_short,cited_case_name,cited_case_name_full,cited_citations
0,98386,https://www.courtlistener.com/opinion/98386/gl...,scotus,Supreme Court of the United States,Gleason,Gleason v. Thaw,Gleason v. Thaw,"[1915 U.S. LEXIS 1780, 59 L. Ed. 717, 35 S. Ct..."
1,98456,https://www.courtlistener.com/opinion/98456/cu...,scotus,Supreme Court of the United States,,Cumberland Glass Manufacturing Co. v. De Witt ...,CUMBERLAND GLASS MANUFACTURING COMPANY v. De W...,"[1915 U.S. LEXIS 1353, 59 L. Ed. 1042, 35 S. C..."
2,131151,https://www.courtlistener.com/opinion/131151/c...,scotus,Supreme Court of the United States,Castro,Castro v. United States,Castro v. United States,"[2003 U.S. LEXIS 9197, 540 U.S. 375, 124 S. Ct..."
3,131161,https://www.courtlistener.com/opinion/131161/g...,scotus,Supreme Court of the United States,Groh,Groh v. Ramirez,GROH v. RAMIREZ Et Al.,"[2004 WL 330057, 2004 U.S. LEXIS 1624, 540 U.S..."
4,262262,https://www.courtlistener.com/opinion/262262/s...,ca8,Court of Appeals for the Eighth Circuit,,State Farm Mutual Automobile Insurance Company...,STATE FARM MUTUAL AUTOMOBILE INSURANCE COMPANY...,"[1963 U.S. App. LEXIS 3672, 324 F.2d 340]"


In [20]:
len(cited_metadata)

15603

# Create result_df by merging citing and cited metadatas

In [21]:
result_df = citing_df.explode("cited_cluster_ids").reset_index(drop=True)
len(result_df)

23082

In [22]:
result_df = result_df.rename(columns={"cited_cluster_ids": "cited_cluster_id"})
result_df.head()

,citing_cluster_id,citing_url,citing_court_id,citing_court_name,opinion_data,cited_cluster_id,opinion_filenames
0,217242,https://www.courtlistener.com/opinion/217242/d...,ca1,Court of Appeals for the First Circuit,"[{'opinion_id': 217242, 'opinion_api': None, '...",2526,[217242_010combined.txt]
1,217242,https://www.courtlistener.com/opinion/217242/d...,ca1,Court of Appeals for the First Circuit,"[{'opinion_id': 217242, 'opinion_api': None, '...",46231,[217242_010combined.txt]
2,217242,https://www.courtlistener.com/opinion/217242/d...,ca1,Court of Appeals for the First Circuit,"[{'opinion_id': 217242, 'opinion_api': None, '...",153651,[217242_010combined.txt]
3,217242,https://www.courtlistener.com/opinion/217242/d...,ca1,Court of Appeals for the First Circuit,"[{'opinion_id': 217242, 'opinion_api': None, '...",179285,[217242_010combined.txt]
4,217242,https://www.courtlistener.com/opinion/217242/d...,ca1,Court of Appeals for the First Circuit,"[{'opinion_id': 217242, 'opinion_api': None, '...",199916,[217242_010combined.txt]


In [23]:
result_df = result_df.merge(cited_metadata, how="left", on="cited_cluster_id")
len(result_df)

23082

In [24]:
result_df.columns

Index(['citing_cluster_id', 'citing_url', 'citing_court_id',
       'citing_court_name', 'opinion_data', 'cited_cluster_id',
       'opinion_filenames', 'cited_url', 'cited_court_id', 'cited_court_name',
       'cited_case_name_short', 'cited_case_name', 'cited_case_name_full',
       'cited_citations'],
      dtype='object')

In [25]:
result_df = result_df[['citing_cluster_id', 'citing_url', 'citing_court_id',
       'citing_court_name', 'opinion_data', 'opinion_filenames', 
       'cited_cluster_id', 'cited_url', 'cited_court_id', 'cited_court_name',
       'cited_case_name_short', 'cited_case_name', 'cited_case_name_full',
       'cited_citations']]

In [26]:
result_df.head()

,citing_cluster_id,citing_url,citing_court_id,citing_court_name,opinion_data,opinion_filenames,cited_cluster_id,cited_url,cited_court_id,cited_court_name,cited_case_name_short,cited_case_name,cited_case_name_full,cited_citations
0,217242,https://www.courtlistener.com/opinion/217242/d...,ca1,Court of Appeals for the First Circuit,"[{'opinion_id': 217242, 'opinion_api': None, '...",[217242_010combined.txt],2526,https://www.courtlistener.com/opinion/2526/isl...,ca2,Court of Appeals for the Second Circuit,,"Island Park, LLC v. CSX Transportation","ISLAND PARK, LLC, Plaintiff-Appellee-Cross-App...","[2009 U.S. App. LEXIS 5219, 559 F.3d 96]"
1,217242,https://www.courtlistener.com/opinion/217242/d...,ca1,Court of Appeals for the First Circuit,"[{'opinion_id': 217242, 'opinion_api': None, '...",[217242_010combined.txt],46231,https://www.courtlistener.com/opinion/46231/ur...,ca5,Court of Appeals for the Fifth Circuit,,Urban Developers LLC v. City of Jackson MS,"URBAN DEVELOPERS LLC, Plaintiff-Appellee, v. C...","[2006 WL 3012860, 2006 U.S. App. LEXIS 26435, ..."
2,217242,https://www.courtlistener.com/opinion/217242/d...,ca1,Court of Appeals for the First Circuit,"[{'opinion_id': 217242, 'opinion_api': None, '...",[217242_010combined.txt],153651,https://www.courtlistener.com/opinion/153651/b...,ca10,Court of Appeals for the Tenth Circuit,Bateman,Bateman v. City of West Bountiful,"Wesley v. BATEMAN, Plaintiff-Appellant, v. CIT...","[1996 WL 384769, 1996 U.S. App. LEXIS 16429, 8..."
3,217242,https://www.courtlistener.com/opinion/217242/d...,ca1,Court of Appeals for the First Circuit,"[{'opinion_id': 217242, 'opinion_api': None, '...",[217242_010combined.txt],179285,https://www.courtlistener.com/opinion/179285/c...,ca1,Court of Appeals for the First Circuit,Cortes-Rivera,Cortés-Rivera v. Department of Corrections & R...,"Enrique CORTÉS-RIVERA, Plaintiff, Appellant, v...","[2010 WL 4608750, 2010 U.S. App. LEXIS 23529, ..."
4,217242,https://www.courtlistener.com/opinion/217242/d...,ca1,Court of Appeals for the First Circuit,"[{'opinion_id': 217242, 'opinion_api': None, '...",[217242_010combined.txt],199916,https://www.courtlistener.com/opinion/199916/d...,ca1,Court of Appeals for the First Circuit,Deniz-Marquez,Deniz v. Municipality of Guaynabo,"Calixto DENIZ, A/K/A Calixto Deniz Marquez, Pl...","[2002 WL 501056, 2002 U.S. App. LEXIS 6425, 28..."


## Tag the target cases from the target courts

In [27]:
result_df.loc[result_df["cited_cluster_id"].isin(target_ids), "cited_target"] = 1
result_df.loc[~result_df["cited_cluster_id"].isin(target_ids), "cited_target"] = 0

## Do some EDA

In [28]:
eda_cols = ['citing_cluster_id', 'citing_court_name', 'cited_cluster_id', 'cited_court_name', 'cited_target']

for col in eda_cols:
    print("----------")
    print(result_df[col].nunique())
    display(result_df[col].value_counts())

----------
1210


citing_cluster_id
4287446    1010
4289638     629
4295814     195
1467978     186
1808186     128
           ... 
7345007       1
7035302       1
7007708       1
7220671       1
7035303       1
Name: count, Length: 1210, dtype: int64

----------
128


citing_court_name
Court of Appeals of Texas                                           4600
Louisiana Court of Appeal                                           3299
Appellate Division of the Supreme Court of the State of New York    1533
Court of Criminal Appeals of Texas                                  1013
Court of Criminal Appeals of Alabama                                 713
                                                                    ... 
Hawaii Supreme Court                                                   3
Appellate Terms of the Supreme Court of New York                       3
Pennsylvania Court of Common Pleas, Dauphin County                     2
Pennsylvania Court of Common Pleas, Lycoming County                    2
Supreme Court of Vermont                                               1
Name: count, Length: 128, dtype: int64

----------
15603


cited_cluster_id
5682590    124
1729419    120
1719177     97
1782914     88
1772937     65
          ... 
1858017      1
1875355      1
1938276      1
7670084      1
91972        1
Name: count, Length: 15603, dtype: int64

----------
229


cited_court_name
Texas Supreme Court                                            3258
Court of Appeals of Texas                                      2382
Louisiana Court of Appeal                                      2166
Supreme Court of the United States                             1421
Supreme Court of Louisiana                                     1309
                                                               ... 
District Court, D. Utah                                           1
U.S. Circuit Court for the District of Eastern Pennsylvania       1
Arizona Tax Court                                                 1
U.S. Circuit Court for the District of Indiana                    1
United States Bankruptcy Court, E.D. New York                     1
Name: count, Length: 229, dtype: int64

----------
2


cited_target
0.0    21872
1.0     1210
Name: count, dtype: int64

In [29]:
df_target = result_df[result_df["cited_target"] == 1]
print(df_target["cited_court_name"].nunique())
df_target["cited_court_name"].value_counts()

42


cited_court_name
New York Court of Appeals                                           195
Supreme Court of Louisiana                                          186
Texas Supreme Court                                                 185
Appellate Division of the Supreme Court of the State of New York     74
District Court of Appeal of Florida                                  54
Louisiana Court of Appeal                                            49
Supreme Court of Rhode Island                                        49
Court of Appeals of Washington                                       30
Court of Criminal Appeals of Alabama                                 28
Supreme Court of Missouri                                            23
Michigan Court of Appeals                                            22
Superior Court of Pennsylvania                                       22
California Court of Appeal                                           20
Supreme Court of Florida                       

# Save the data for future use

In [30]:
result_df.to_json("data/state_citing_cited.json")